In [1]:
import requests
from bs4 import BeautifulSoup
import time
from pymongo import MongoClient
from datetime import datetime

# ============================================
# الإعدادات (تعديل للـ Connection الجديد المنفصل)
# ============================================
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/119.0.0.0 Safari/537.36"
    )
}

BASE_URL = "https://www.nhsinform.scot/illnesses-and-conditions/a-to-z/"

# الاتصال بـ MongoDB المحلي
client = MongoClient("mongodb://localhost:27017/")
db = client["medical_ai_db"]

# 📌 تم تغيير الأسماء هنا لتسجيل الداتا المتساوية في مكان جديد تماماً دون مسح القديم
conditions_col = db["conditions_structured"]          # الكوليكشن الجديد للداتا المتساوية
checkpoint_col = db["scrape_checkpoint_structured"]   # الـ Checkpoint الجديد المستقل

# منع التكرار في الكوليكشن الجديد بناءً على اسم المرض
conditions_col.create_index("condition", unique=True)


# ============================================
# دوال المساعدة لتوحيد وتساوي الـ Features
# ============================================
def fetch_html(url, retries=2, timeout=10):
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout)
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "html.parser")
            else:
                print(f"[warn] HTTP {resp.status_code} → {url}")
                return None
        except requests.RequestException as e:
            if attempt < retries - 1:
                time.sleep(1)
                continue
            print(f"[skip] فشل نهائيًا: {url}")
            return None
    return None


def classify_section(heading_text):
    text = heading_text.lower()

    # دمج الكلمات التحذيرية والعلاجية والتوصيات في تصنيف واحد لتوحيد العواميد
    recommendation_keywords = [
        "treat", "manag", "prevent", "self-help", "living with",
        "support", "exercise", "diet", "medication",
        "what you can do", "lifestyle", "help and support",
        "when to", "emergency", "seek", "urgent",
        "complication", "warning", "999", "a&e"
    ]
    symptom_keywords = [
        "symptom", "sign", "what is", "about", "overview"
    ]
    cause_keywords = [
        "cause", "why", "risk factor", "diagnos"
    ]

    if any(kw in text for kw in recommendation_keywords):
        return "recommendations"
    if any(kw in text for kw in cause_keywords):
        return "causes"
    if any(kw in text for kw in symptom_keywords):
        return "symptoms"

    return "causes"  # fallback


def scrape_condition_page(url, condition_name):
    soup = fetch_html(url)
    if not soup:
        return None

    # الهيكل الثابت والمتساوي 100% لكل الأمراض (3 حقول أساسية بجانب الاسم والـ URL)
    data = {
        "condition": condition_name,
        "url": url,
        "symptoms": [],
        "causes": [],
        "recommendations": [], 
        "scraped_at": datetime.utcnow(),
        "source": "NHS Inform"
    }

    headings = soup.select("h2.wp-block-heading")
    if not headings:
        headings = soup.find_all("h2")

    for h2 in headings:
        h2_text = h2.get_text(strip=True)
        category = classify_section(h2_text)

        paragraphs = []
        sibling = h2.find_next_sibling()
        while sibling and sibling.name != "h2":
            if sibling.name == "p":
                text = sibling.get_text(strip=True)
                if text:
                    paragraphs.append(text)
            elif sibling.name == "ul":
                for li in sibling.find_all("li"):
                    li_text = li.get_text(strip=True)
                    if li_text:
                        paragraphs.append(li_text)
            sibling = sibling.find_next_sibling()

        data[category].extend(paragraphs)

    # تحويل القوائم لنصوص، وكتابة "not mentioned" للـ Features الغائبة للحفاظ على التساوي
    for field in ["symptoms", "causes", "recommendations"]:
        if data[field]:
            data[field] = " ".join(data[field])
        else:
            data[field] = "not mentioned"

    return data


def save_condition(data):
    try:
        conditions_col.update_one(
            {"condition": data["condition"]},
            {"$set": data},
            upsert=True
        )
        return True
    except Exception as e:
        print(f"[error] فشل الحفظ: {e}")
        return False


def save_checkpoint(index, condition_name):
    checkpoint_col.update_one(
        {"_id": "scraper_checkpoint"},
        {"$set": {
            "last_index": index,
            "last_condition": condition_name,
            "updated_at": datetime.utcnow()
        }},
        upsert=True
    )


def get_checkpoint():
    cp = checkpoint_col.find_one({"_id": "scraper_checkpoint"})
    if cp:
        print(f"📌 استكمال من الـ Checkpoint الجديد: [{cp['last_index']}] {cp['last_condition']}")
        return cp["last_index"]
    return 0


def get_all_condition_links():
    print("📋 جلب قائمة الأمراض من صفحة A-Z...")
    soup = fetch_html(BASE_URL)
    if not soup:
        print("❌ فشل تحميل صفحة A-Z")
        return []

    seen = set()
    links = []
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        text = a.get_text(strip=True)
        if "/illnesses-and-conditions/" in href and href not in seen and text:
            part = href.split("/illnesses-and-conditions/")[-1]
            depth = len([p for p in part.split("/") if p])
            if depth >= 2:
                seen.add(href)
                links.append({"name": text, "url": href})

    print(f"✅ تم العثور على {len(links)} رابط مرض")
    return links


# ============================================
# الـ Main — تشغيل الـ Scraper
# ============================================
def run_scraper(delay=1.5):
    links = get_all_condition_links()
    if not links:
        return

    # جلب الـ Checkpoint الجديد (هيبدأ من 0 بما إنه كوليكشن جديد لأول مرة)
    start_index = get_checkpoint()
    remaining = links[start_index:]

    print(f"\n🚀 بدء السحب بالهيكل المتساوي الجديد من index {start_index}...")
    print(f"   المتبقي: {len(remaining)} من {len(links)} مرض\n")

    success = 0
    failed = 0

    for i, link in enumerate(remaining, start=start_index):
        name = link["name"]
        url = link["url"]

        print(f"[{i+1}/{len(links)}] {name}", end=" ... ")

        data = scrape_condition_page(url, name)

        # التحقق من وجود محتوى حقيقي تم سحبه
        has_content = data and (
            data["symptoms"] != "not mentioned" or
            data["causes"] != "not mentioned" or
            data["recommendations"] != "not mentioned"
        )

        if has_content:
            if save_condition(data):
                print("✅ تم الحفظ (هيكل متساوي)")
                success += 1
            else:
                print("❌ فشل الحفظ في الـ DB")
                failed += 1
        else:
            print("⚠️ الصفحة فارغة أو بدون محتوى مطابق")
            failed += 1

        # حفظ الـ Checkpoint الجديد بعد كل مرض لمنع الضياع
        save_checkpoint(i + 1, name)

        time.sleep(delay)

    print(f"\n{'='*55}")
    print(f"✅ تم بنجاح: {success}  |  ⚠️ فشل أو فارغ: {failed}")
    print(f"📦 إجمالي المستندات في الكوليكشن الجديد (conditions_structured): {conditions_col.count_documents({})}")
    print(f"{'='*55}")


if __name__ == "__main__":
    # تأكد من تشغيل الـ MongoDB أولاً على جهازك قبل تشغيل السكريبت
    run_scraper(delay=1.5)

📋 جلب قائمة الأمراض من صفحة A-Z...
✅ تم العثور على 423 رابط مرض

🚀 بدء السحب بالهيكل المتساوي الجديد من index 0...
   المتبقي: 423 من 423 مرض

[1/423] Abdominal aortic aneurysm ... 

C:\Users\User\AppData\Local\Temp\ipykernel_24964\2652915682.py:93: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "scraped_at": datetime.utcnow(),
C:\Users\User\AppData\Local\Temp\ipykernel_24964\2652915682.py:150: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "updated_at": datetime.utcnow()


✅ تم الحفظ (هيكل متساوي)
[2/423] About aplastic anaemia ... ✅ تم الحفظ (هيكل متساوي)
[3/423] Achilles tendinopathy ... ✅ تم الحفظ (هيكل متساوي)
[4/423] Acne ... ✅ تم الحفظ (هيكل متساوي)
[5/423] Acute cholecystitis ... ✅ تم الحفظ (هيكل متساوي)
[6/423] Acute lymphoblastic leukaemia ... ✅ تم الحفظ (هيكل متساوي)
[7/423] Acute myeloid leukaemia ... ✅ تم الحفظ (هيكل متساوي)
[8/423] Acute pancreatitis ... ✅ تم الحفظ (هيكل متساوي)
[9/423] Acute respiratory infection (ARI) ... ✅ تم الحفظ (هيكل متساوي)
[10/423] Addison’s disease ... ✅ تم الحفظ (هيكل متساوي)
[11/423] Alcohol-related liver disease ... ✅ تم الحفظ (هيكل متساوي)
[12/423] Allergic rhinitis ... ✅ تم الحفظ (هيكل متساوي)
[13/423] Allergies ... ✅ تم الحفظ (هيكل متساوي)
[14/423] Alopecia (hair loss) ... ✅ تم الحفظ (هيكل متساوي)
[15/423] Alzheimer’s disease ... ✅ تم الحفظ (هيكل متساوي)
[16/423] Anal cancer ... ✅ تم الحفظ (هيكل متساوي)
[17/423] Anaphylaxis ... ✅ تم الحفظ (هيكل متساوي)
[18/423] Angina ... ✅ تم الحفظ (هيكل متساوي)
[19/423] Ang

In [2]:
import pandas as pd
from pymongo import MongoClient

# الاتصال بـ MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["medical_ai_db"]
conditions_col = db["conditions_structured"]

# سحب الداتا الجديدة المتساوية
data_cursor = conditions_col.find({}, {
    "_id": 0, 
    "condition": 1, 
    "symptoms": 1, 
    "causes": 1, 
    "recommendations": 1
})

# تحويلها لـ DataFrame وحفظها كـ CSV
df_clean = pd.DataFrame(list(data_cursor))
df_clean.to_csv('structured_medical_dataset.csv', index=False)

print("✅ تم تحويل الداتا المتساوية بنجاح لملف: structured_medical_dataset.csv")
print(f"أبعاد الداتا الجديدة: {df_clean.shape}")
print(df_clean.head())

✅ تم تحويل الداتا المتساوية بنجاح لملف: structured_medical_dataset.csv
أبعاد الداتا الجديدة: (379, 4)
                   condition  \
0  Abdominal aortic aneurysm   
1     About aplastic anaemia   
2      Achilles tendinopathy   
3                       Acne   
4        Acute cholecystitis   

                                              causes  \
0  It’s not known exactly what causes the aortic ...   
1  Aplastic anaemia can be: very severe severe no...   
2  Achilles tendinopathy can occur in both active...   
3  Speak to your pharmacist if you have symptoms ...   
4  The causes of acute cholecystitis can be group...   

                                     recommendations  \
0  If a large AAA is detected before it ruptures,...   
1  If you have aplastic anaemia, your treatment w...   
2  Managing Achilles tendinopathy can take time, ...   
3  Treatment for acne depends on how severe it is...   
4  It’s important for acute cholecystitis to be d...   

                               

In [3]:
import pandas as pd
from pymongo import MongoClient

try:
    # 1. الاتصال بـ MongoDB المحلي
    client = MongoClient("mongodb://localhost:27017/")
    db = client["medical_ai_db"]
    conditions_col = db["conditions_structured"]

    # 2. سحب الداتا المتساوية الجديدة (379 مرض) مع استبعاد الـ ID بتاع مونجو
    print("📦 جلب البيانات المتساوية من MongoDB...")
    data_cursor = conditions_col.find({}, {
        "_id": 0, 
        "condition": 1, 
        "symptoms": 1, 
        "causes": 1, 
        "recommendations": 1
    })

    # 3. تحويل البيانات إلى DataFrame
    df_clean = pd.DataFrame(list(data_cursor))

    # التأكد من أن الكوليكشن مش فاضي
    if not df_clean.empty:
        # 4. حفظ الداتا كـ ملف CSV
        output_filename = 'structured_medical_dataset.csv'
        df_clean.to_csv(output_filename, index=False)
        
        print(f"\n✅ مبروك يا هندسة! الملف نزل بنجاح باسم: {output_filename}")
        print(f"📊 أبعاد الـ CSV الجديد: {df_clean.shape} (379 مرض، و 4 أعمدة متساوية تماماً)")
    else:
        print("⚠️ تحذير: الكوليكشن 'conditions_structured' فاضي! تأكد من اسم الـ DB والـ Collection.")

except Exception as e:
    print(f"❌ حدث خطأ أثناء الاتصال أو التصدير: {e}")

📦 جلب البيانات المتساوية من MongoDB...

✅ مبروك يا هندسة! الملف نزل بنجاح باسم: structured_medical_dataset.csv
📊 أبعاد الـ CSV الجديد: (379, 4) (379 مرض، و 4 أعمدة متساوية تماماً)


In [4]:
df=pd.read_csv('structured_medical_dataset.csv')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 379 entries, 0 to 378
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   condition        379 non-null    str  
 1   causes           379 non-null    str  
 2   recommendations  379 non-null    str  
 3   symptoms         379 non-null    str  
dtypes: str(4)
memory usage: 12.0 KB


In [2]:
pip install google-genai pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ---------------------------------- ----- 786.4/914.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 1.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 1.4 MB/s eta 0:00:02
   --------- ------------------------------ 0.5/2.2 MB 1.4 MB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.2 MB 922.5 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.2 MB 982.1 kB/s eta 0:00:02
   ----------------------- ---------------- 1.3/2.2 MB 1.1 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.2 MB 1.1 MB/s eta 0:00:01
   ---------------------------

In [6]:
import pandas as pd
import json
import os
import time
from google import genai
from google.genai import types
from tqdm.notebook import tqdm  # tqdm مخصصة للـ Notebooks شكلها أشيك

# 1. إعداد الـ Client بالـ API Key بتاعك مباشرة
GEMINI_API_KEY = "حط_الـ_API_KEY_بتاعك_هنا"
client = genai.Client(api_key=GEMINI_API_KEY)

INPUT_FILE = 'structured_medical_dataset.csv'
OUTPUT_FILE = 'augmented_medical_dataset.csv'

# 2. تحميل البيانات وتفقّد التقدّم السابق
df = pd.read_csv(INPUT_FILE)

existing_augmented_data = []
processed_conditions = set()

if os.path.exists(OUTPUT_FILE):
    try:
        existing_df = pd.read_csv(OUTPUT_FILE)
        existing_augmented_data = existing_df.to_dict('records')
        processed_conditions = set(existing_df['condition'].unique())
        print(f"📌 كملنا على الشغل القديم! تم معالجة {len(processed_conditions)} مرض سابقاً.")
    except Exception as e:
        print(f"⚠️ جاري البدء من الجديد: {e}")

# 3. الـ Prompt الهيكلي
prompt_template = """
You are a medical dataset generator. Given the medical symptoms of a condition, generate 5 distinct, realistic patient queries (how a real person might describe their symptoms to a doctor or a medical chatbot).

Include diverse styles across the 5 queries:
1. Casual & simple description
2. Anxious/urgent tone
3. Short search-like phrase
4. Detailed paragraph describing daily impact
5. Direct question format

Return ONLY a valid JSON object with a single key "queries" which contains an array of 5 strings.

Condition Name: {condition}
Symptoms: {symptoms}
"""

# 4. التوليد وحفظ التقدم تلقائياً
augmented_rows = existing_augmented_data
batch_size = 10 

print("🚀 بدء عملية الـ Augmentation...")

for index, row in tqdm(df.iterrows(), total=len(df)):
    condition = row['condition']
    symptoms = row['symptoms']
    causes = row['causes']
    recommendations = row['recommendations']

    if condition in processed_conditions:
        continue

    symptoms_text = f"General query regarding {condition}" if symptoms == "not mentioned" else symptoms

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt_template.format(condition=condition, symptoms=symptoms_text),
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.7
            ),
        )

        extracted_json = json.loads(response.text)
        queries = extracted_json.get("queries", [])

        for q in queries:
            augmented_rows.append({
                'condition': condition,
                'patient_query': q,
                'symptoms': symptoms,
                'causes': causes,
                'recommendations': recommendations
            })

        processed_conditions.add(condition)

        # حفظ تلقائي كل 10 أمراض
        if len(processed_conditions) % batch_size == 0:
            pd.DataFrame(augmented_rows).to_csv(OUTPUT_FILE, index=False)

        time.sleep(0.4)

    except Exception as e:
        print(f"\n⚠️ خطأ عند المرض [{condition}]: {e}")
        time.sleep(2)

# حفظ نهائي
pd.DataFrame(augmented_rows).to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ كلو تمام يا جنرال!")
print(f"📦 إجمالي الأمراض المعالجة: {len(processed_conditions)}")
print(f"📊 أبعاد الداتا الجدية: {len(augmented_rows)} سطر!")

🚀 بدء عملية الـ Augmentation...


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html